# HealthAI Disease Classifier — Synthetic Dataset (Bulletproof Build)
## Bio_ClinicalBERT · 12 diseases · 7200 balanced samples

### Root cause of 'always dengue' fixed:
- **Gradual unfreezing**: Phase 1 trains only the classifier head (fast, stable).
  Phase 2 unfreezes all layers for full fine-tuning.
- **Training verification**: loss is checked after step 1 — if it doesn't drop below
  the random baseline (2.48), training halts with a clear error message.
- **Diversity assertion**: after training, checks that all 12 classes appear in predictions.
- **Single execution block**: train + load best weights all in one cell — can't be skipped.
- **Epoch checkpoints**: model saved every epoch, not just 'best', so nothing is lost
  on Colab disconnect.

**Before running:** Runtime → Change runtime type → T4 GPU
**Upload both files** to the Colab file browser before running.
### Step 5b (new) — TF-IDF + Cosine Similarity disambiguation layer:
- Builds a **per-disease TF-IDF centroid** from training data (document-term matrix).
- At inference, computes **cosine similarity** between query and every disease centroid.
- Blends with BERT when BERT is uncertain or predicts dengue with moderate confidence.
- Generic inputs like *"fever and headache"* no longer collapse to dengue because
  dengue-specific markers are absent → cosine similarity drops → correct redistribution.


## Step 1 — Install & imports


In [ ]:
!pip install transformers torch scikit-learn matplotlib seaborn --quiet
print('Libraries ready.')


## Step 2 — Configuration


In [ ]:
import os, json, random, platform
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# ── Paths ─────────────────────────────────────────────────────────────────
CSV_PATH   = 'healthai_synthetic_training_set_7200.csv'
JSONL_PATH = 'healthai_synthetic_training_set_7200.jsonl'

# ── Hyperparameters ───────────────────────────────────────────────────────
MODEL_NAME    = 'emilyalsentzer/Bio_ClinicalBERT'
MAX_LEN       = 128
BATCH_SIZE    = 32
PHASE1_EPOCHS = 2      # classifier-only warmup
PHASE2_EPOCHS = 3      # full fine-tune
LR_HEAD       = 1e-3   # higher LR for classifier head only
LR_FULL       = 2e-5   # standard BERT fine-tune LR
PATIENCE      = 2

# ── TF-IDF disambiguation config ──────────────────────────────────────────
TFIDF_BLEND_ALPHA   = 0.65   # BERT weight in blend  (1-α goes to TF-IDF)
AMBIGUITY_THRESHOLD = 0.55   # apply blend when BERT top-prob < this
DENGUE_OVERRIDE_CAP = 0.72   # also blend when dengue pred < this confidence
SEED          = 42

# ── Label map ─────────────────────────────────────────────────────────────
DISEASE_LABELS = {
    'epilepsy':0, 'migraine':1, 'stroke':2, 'diabetic_neuropathy':3,
    'dementia':4, 'parkinsons':5, 'dengue':6, 'malaria':7,
    'kala_azar':8, 'chikungunya':9, 'japanese_encephalitis':10, 'scrub_typhus':11,
}
LABEL_NAMES = {v: k for k, v in DISEASE_LABELS.items()}
NUM_LABELS  = len(DISEASE_LABELS)
RANDOM_BASELINE_LOSS = float(np.log(NUM_LABELS))  # 2.485 — untrained model threshold

# ── Seeds ─────────────────────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'Config ready. {NUM_LABELS} classes. Random baseline loss = {RANDOM_BASELINE_LOSS:.3f}')


## Step 3 — GPU check


In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if str(DEVICE) != 'cuda':
    raise RuntimeError(
        'GPU not detected!\n'
        'Go to: Runtime → Change runtime type → Hardware accelerator → T4 GPU\n'
        'Then: Runtime → Restart and run all'
    )
torch.cuda.manual_seed_all(SEED)
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


## Step 4 — Load & validate dataset


In [ ]:
if os.path.exists(CSV_PATH):
    df_raw = pd.read_csv(CSV_PATH)
    print(f'Loaded CSV: {len(df_raw)} rows')
elif os.path.exists(JSONL_PATH):
    df_raw = pd.DataFrame([json.loads(l) for l in open(JSONL_PATH)])
    print(f'Loaded JSONL: {len(df_raw)} rows')
else:
    raise FileNotFoundError(
        f'Neither {CSV_PATH} nor {JSONL_PATH} found.\n'
        'Upload the files to the Colab file browser (left panel) and re-run.'
    )

assert 'label' in df_raw.columns and 'input_text' in df_raw.columns
df_raw = df_raw[df_raw['label'].isin(DISEASE_LABELS)].copy()
df_raw['label_id']   = df_raw['label'].map(DISEASE_LABELS)
df_raw['input_text'] = df_raw['input_text'].astype(str).str.strip()

print(f'Valid samples : {len(df_raw)}')
print(f'Classes       : {df_raw["label"].nunique()}')
print()
print('Samples per class:')
print(df_raw['label'].value_counts().to_string())
assert df_raw['label'].nunique() == NUM_LABELS, 'Missing disease classes in data!'
print('\nData validated ✓')


## Step 5 — Train / val / test split (70 / 15 / 15)


In [ ]:
train_df, temp_df = train_test_split(
    df_raw, test_size=0.30, random_state=SEED, stratify=df_raw['label_id'])
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df['label_id'])

print(f'Train : {len(train_df):5d}  Val : {len(val_df):5d}  Test : {len(test_df):5d}')
print()
print('Train class distribution:')
print(train_df['label'].value_counts().to_string())


## Step 5b — TF-IDF Disease Profiler (dengue-bias fix)

**Root cause** of the *always-dengue* problem: Bio_ClinicalBERT collapses to dengue for
generic symptoms because dengue shares the most surface symptoms with other diseases
and dominates uncertain predictions.

**Fix with TF-IDF + cosine similarity (vector space model):**

1. `TfidfVectorizer` fitted on **training texts only** → builds a (N × vocab) document-term matrix.
2. Each disease class → **mean TF-IDF vector (centroid)**.  High-IDF disease-specific terms
   (`NS1`, `platelet`, `rK39`, `levodopa`, `eschar`) dominate their respective centroids.
3. At predict time: query → TF-IDF vector → **cosine similarity** with each disease centroid.
4. When BERT is uncertain (`top_conf < AMBIGUITY_THRESHOLD`) or predicts dengue with moderate
   confidence, **blend**: `p_final = α×p_BERT + (1-α)×p_TF-IDF`

**Why this fixes the bias:** `fever headache` has near-zero cosine similarity with the dengue
centroid (because `NS1`, `thrombocytopenia`, `retro-orbital` are absent) → probability
redistributes to more plausible diseases rather than defaulting to dengue.

In [ ]:
# ── TF-IDF Disease Profiler ──────────────────────────────────────────────────────
# Fitted on train_df ONLY (no val/test leakage).
# Provides cosine-similarity-based disease probabilities blended with BERT
# when BERT is uncertain or when it over-predicts dengue.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as _cosine_sim


class TFIDFDiseaseProfiler:
    """
    Sparse vector-space disease profiler.

    Each disease is represented as the centroid (mean TF-IDF vector) of all
    its training documents in the document-term matrix.

    High-IDF (disease-specific) words dominate each centroid:
      dengue     -> NS1, platelet, retro-orbital, thrombocytopenia, leucopenia
      malaria    -> vivax, trophozoites, pLDH, RDT, parasite
      kala_azar  -> rK39, splenomegaly, visceral, leishmaniasis
      scrub      -> eschar, weil-felix, OXK, scrub, orientia

    Generic terms like 'fever', 'headache' have LOW IDF and barely move
    any centroid -- so they cannot drive a dengue false-positive.
    """

    def __init__(self, max_features=8000, ngram_range=(1, 2)):
        self.vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=ngram_range,
            sublinear_tf=True,    # 1+log(TF): dampens very frequent terms
            min_df=2,             # ignore terms in < 2 documents
            stop_words='english',
        )
        self.centroids     = {}   # {label_id: (1, V) ndarray}
        self.feature_names = None

    def fit(self, df):
        """Fit on training split ONLY -- no data leakage."""
        texts        = df['input_text'].tolist()
        tfidf_matrix = self.vectorizer.fit_transform(texts)         # (N, V) sparse
        self.feature_names = self.vectorizer.get_feature_names_out()

        for label_id in range(NUM_LABELS):
            mask     = (df['label_id'].values == label_id)
            centroid = np.asarray(tfidf_matrix[mask].mean(axis=0))  # (1, V)
            self.centroids[label_id] = centroid

        V = len(self.feature_names)
        print(f'TF-IDF profiler fitted  |  vocab = {V:,}  |  {NUM_LABELS} centroids')
        print()
        print('Top-5 TF-IDF signature terms per disease (highest centroid weight):')
        for lid in range(NUM_LABELS):
            top_idx   = np.argsort(self.centroids[lid][0])[-5:][::-1]
            top_terms = [self.feature_names[i] for i in top_idx]
            print(f'  {LABEL_NAMES[lid]:25s}: {", ".join(top_terms)}')

    def score(self, text):
        """
        Cosine similarity of query text against every disease centroid.

        Returns
        -------
        sims : np.ndarray, shape (NUM_LABELS,), normalised to sum = 1.
        """
        vec  = self.vectorizer.transform([text])           # (1, V) sparse
        sims = np.zeros(NUM_LABELS, dtype=np.float64)
        for lid, centroid in self.centroids.items():
            sims[lid] = float(_cosine_sim(vec, centroid)[0, 0])
        total = sims.sum()
        if total > 1e-10:
            sims /= total          # normalise so values sum to 1
        return sims


# Fit on training split only
tfidf_profiler = TFIDFDiseaseProfiler(max_features=8000, ngram_range=(1, 2))
tfidf_profiler.fit(train_df)
print()
print('TF-IDF profiler ready ✓')

## Step 6 — Tokenize & DataLoaders


In [ ]:
print(f'Loading tokenizer from {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class DiseaseDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts  = list(texts)
        self.labels = list(labels)
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        enc = tokenizer(
            str(self.texts[i]),
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[i], dtype=torch.long)
        }

_nw = 0 if platform.system() == 'Windows' else 2

train_ds = DiseaseDataset(train_df['input_text'], train_df['label_id'])
val_ds   = DiseaseDataset(val_df['input_text'],   val_df['label_id'])
test_ds  = DiseaseDataset(test_df['input_text'],  test_df['label_id'])

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=_nw)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=_nw)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=_nw)

print(f'Train batches: {len(train_dl)}  Val: {len(val_dl)}  Test: {len(test_dl)}')
print(f'MAX_LEN={MAX_LEN}  BATCH_SIZE={BATCH_SIZE}  num_workers={_nw}')


## Step 7 — Load Bio_ClinicalBERT


In [ ]:
print(f'Loading {MODEL_NAME}...')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    ignore_mismatched_sizes=True
)
model = model.to(DEVICE)
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable_params:,}')


## Step 8 — Training helpers


In [ ]:
def run_epoch(model, dl, criterion, optimizer=None, scheduler=None, phase='train'):
    """Run one epoch. If optimizer is None, runs eval mode."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    first_step_loss = None

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for step, batch in enumerate(dl):
            ids   = batch['input_ids'].to(DEVICE)
            mask  = batch['attention_mask'].to(DEVICE)
            lbls  = batch['label'].to(DEVICE)

            out  = model(input_ids=ids, attention_mask=mask)
            loss = criterion(out.logits, lbls)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                # Gradient norm — confirms backprop is working
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                if scheduler: scheduler.step()

            total_loss += loss.item()
            preds       = torch.argmax(out.logits, dim=1)
            correct    += (preds == lbls).sum().item()
            total      += lbls.size(0)

            if step == 0:
                first_step_loss = loss.item()
                if is_train:
                    print(f'  [{phase}] Step 0 loss={loss.item():.4f}  '
                          f'grad_norm={grad_norm:.4f}  '
                          f'(random baseline={RANDOM_BASELINE_LOSS:.3f})')

    avg_loss = total_loss / len(dl)
    avg_acc  = correct / total * 100
    return avg_loss, avg_acc, first_step_loss


def make_criterion(train_df):
    counts = train_df['label_id'].value_counts().sort_index()
    w = torch.tensor([1.0 / counts[i] for i in range(NUM_LABELS)], dtype=torch.float).to(DEVICE)
    w = w / w.sum() * NUM_LABELS
    return nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)

criterion = make_criterion(train_df)
print('Criterion ready.')


## Step 9 — Phase 1: Train classifier head only

Freeze all BERT layers. Train only the final classification layer for 2 epochs.
This is fast (~2 min/epoch on T4) and ensures the classifier learns good initial weights
before the full model is unfrozen.


In [ ]:
# ── Freeze all BERT encoder layers ────────────────────────────────────────
for name, param in model.named_parameters():
    if 'classifier' not in name:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Phase 1: {trainable:,} trainable params (classifier head only)')

optimizer_p1 = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR_HEAD, weight_decay=0.01
)
total_steps_p1  = len(train_dl) * PHASE1_EPOCHS
scheduler_p1    = get_linear_schedule_with_warmup(
    optimizer_p1, num_warmup_steps=int(0.1 * total_steps_p1),
    num_training_steps=total_steps_p1
)

p1_train_losses, p1_train_accs = [], []
p1_val_losses,   p1_val_accs   = [], []

print(f'\n=== Phase 1: Classifier warmup ({PHASE1_EPOCHS} epochs) ===')
for epoch in range(1, PHASE1_EPOCHS + 1):
    tr_loss, tr_acc, step0_loss = run_epoch(
        model, train_dl, criterion, optimizer_p1, scheduler_p1, phase=f'P1-Ep{epoch}')

    # Safety check: loss must be below random baseline after step 0
    if epoch == 1 and step0_loss is not None and step0_loss >= RANDOM_BASELINE_LOSS:
        raise RuntimeError(
            f'Step 0 loss={step0_loss:.4f} >= random baseline={RANDOM_BASELINE_LOSS:.3f}.\n'
            'Training is not working. Check GPU, restart runtime, and try again.'
        )

    vl_loss, vl_acc, _ = run_epoch(model, val_dl, criterion, phase=f'P1-Val{epoch}')
    p1_train_losses.append(tr_loss); p1_train_accs.append(tr_acc)
    p1_val_losses.append(vl_loss);   p1_val_accs.append(vl_acc)

    print(f'Phase1 Ep {epoch}/{PHASE1_EPOCHS} '
          f'| Train loss={tr_loss:.4f} acc={tr_acc:.1f}% '
          f'| Val loss={vl_loss:.4f} acc={vl_acc:.1f}%')
    torch.save(model.state_dict(), f'checkpoint_p1_ep{epoch}.pt')

print('Phase 1 complete ✓')


## Step 10 — Phase 2: Full fine-tuning (all layers unfrozen)


In [ ]:
# ── Unfreeze all layers ────────────────────────────────────────────────────
for param in model.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Phase 2: {trainable:,} trainable params (all layers)')

optimizer_p2 = torch.optim.AdamW(
    model.parameters(), lr=LR_FULL, weight_decay=0.01
)
total_steps_p2 = len(train_dl) * PHASE2_EPOCHS
scheduler_p2   = get_linear_schedule_with_warmup(
    optimizer_p2, num_warmup_steps=int(0.06 * total_steps_p2),
    num_training_steps=total_steps_p2
)

p2_train_losses, p2_train_accs = [], []
p2_val_losses,   p2_val_accs   = [], []
best_val_loss = float('inf')
best_epoch    = 0
patience_ctr  = 0

print(f'\n=== Phase 2: Full fine-tuning ({PHASE2_EPOCHS} epochs) ===')
for epoch in range(1, PHASE2_EPOCHS + 1):
    tr_loss, tr_acc, _ = run_epoch(
        model, train_dl, criterion, optimizer_p2, scheduler_p2, phase=f'P2-Ep{epoch}')
    vl_loss, vl_acc, _ = run_epoch(model, val_dl, criterion, phase=f'P2-Val{epoch}')

    p2_train_losses.append(tr_loss); p2_train_accs.append(tr_acc)
    p2_val_losses.append(vl_loss);   p2_val_accs.append(vl_acc)

    print(f'Phase2 Ep {epoch}/{PHASE2_EPOCHS} '
          f'| Train loss={tr_loss:.4f} acc={tr_acc:.1f}% '
          f'| Val loss={vl_loss:.4f} acc={vl_acc:.1f}%')

    # Save every epoch + track best
    torch.save(model.state_dict(), f'checkpoint_p2_ep{epoch}.pt')
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        best_epoch    = epoch
        patience_ctr  = 0
        torch.save(model.state_dict(), 'best_model.pt')
        print(f'  ✓ Best saved (val_loss={vl_loss:.4f})')
    else:
        patience_ctr += 1
        print(f'  No improvement ({patience_ctr}/{PATIENCE})')
        if patience_ctr >= PATIENCE:
            print(f'Early stopping at phase2 epoch {epoch}.')
            break

# Load best weights — IN THIS SAME CELL so it cannot be skipped
model.load_state_dict(torch.load('best_model.pt'))
TRAINING_COMPLETE = True
print(f'\nBest model (phase2 epoch {best_epoch}) loaded. Training complete ✓')


## Step 11 — Training sanity check


In [ ]:
assert TRAINING_COMPLETE, 'Training did not complete! Run Steps 9-10 first.'

# Quick pass over val set — assert all 12 classes appear in predictions
model.eval()
quick_preds = []
with torch.no_grad():
    for batch in val_dl:
        out = model(input_ids=batch['input_ids'].to(DEVICE),
                    attention_mask=batch['attention_mask'].to(DEVICE))
        quick_preds.extend(torch.argmax(out.logits, dim=1).cpu().tolist())

unique_classes_predicted = len(set(quick_preds))
print(f'Unique classes predicted on val set: {unique_classes_predicted} / {NUM_LABELS}')

if unique_classes_predicted < NUM_LABELS:
    missing = set(range(NUM_LABELS)) - set(quick_preds)
    missing_names = [LABEL_NAMES[m] for m in missing]
    print(f'WARNING: Model never predicts: {missing_names}')
    print('The model may not have trained properly. Consider re-running Steps 9-10.')
else:
    print('All 12 disease classes predicted ✓  — no class collapse detected.')

from collections import Counter
dist = Counter(quick_preds)
print('\nPrediction distribution on val set:')
for i in range(NUM_LABELS):
    bar = '█' * (dist.get(i,0) // 3)
    print(f'  {LABEL_NAMES[i]:25s}: {dist.get(i,0):4d}  {bar}')


## Step 12 — Evaluate on held-out test set


In [ ]:
assert TRAINING_COMPLETE, 'Training did not complete! Run Steps 9-10 first.'

model.eval()
all_preds, all_labels_list = [], []
with torch.no_grad():
    for batch in test_dl:
        out = model(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE)
        )
        all_preds.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
        all_labels_list.extend(batch['label'].numpy())

acc = accuracy_score(all_labels_list, all_preds)
f1m = f1_score(all_labels_list, all_preds, average='macro')
f1w = f1_score(all_labels_list, all_preds, average='weighted')
label_list = [LABEL_NAMES[i] for i in range(NUM_LABELS)]

print('=' * 58)
print('HealthAI — Test Set Results')
print('=' * 58)
print(f'Accuracy   : {acc*100:.2f}%')
print(f'F1 macro   : {f1m:.4f}')
print(f'F1 weighted: {f1w:.4f}')
print()
print(classification_report(all_labels_list, all_preds, target_names=label_list))


## Step 13 — Confusion matrix & training curves


In [ ]:
cm = confusion_matrix(all_labels_list, all_preds)
plt.figure(figsize=(14, 11))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=label_list, yticklabels=label_list, cmap='Blues')
plt.title('HealthAI — Confusion Matrix', fontsize=13, fontweight='bold')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight'); plt.show()

# Combine phase 1 + phase 2 curves
all_train_losses = p1_train_losses + p2_train_losses
all_val_losses   = p1_val_losses   + p2_val_losses
all_train_accs   = p1_train_accs   + p2_train_accs
all_val_accs     = p1_val_accs     + p2_val_accs
epochs_ran       = len(all_train_losses)
phase_boundary   = PHASE1_EPOCHS + 0.5

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
x = range(1, epochs_ran + 1)
axes[0].plot(x, all_train_losses, 'o-', color='firebrick',  label='Train', linewidth=2)
axes[0].plot(x, all_val_losses,   's--', color='steelblue', label='Val',   linewidth=2)
axes[0].axvline(phase_boundary, color='gray', linestyle=':', label='Unfreeze')
axes[0].set_title('Loss', fontweight='bold'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(x, all_train_accs, 'o-', color='firebrick',  label='Train', linewidth=2)
axes[1].plot(x, all_val_accs,   's--', color='steelblue', label='Val',   linewidth=2)
axes[1].axvline(phase_boundary, color='gray', linestyle=':', label='Unfreeze')
axes[1].set_title('Accuracy (%)', fontweight='bold'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('HealthAI Training Curves (Phase1=head only | Phase2=full)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight'); plt.show()
print('Charts saved.')


## Step 14 — Live prediction demo


In [ ]:
assert TRAINING_COMPLETE, 'Training did not complete! Run Steps 9-10 first.'


def predict(text, use_tfidf=True):
    """
    Predict disease from a free-text symptom description.

    TF-IDF disambiguation (Step 5b) is blended with BERT softmax when:
      (a) BERT top-class probability < AMBIGUITY_THRESHOLD  -- model uncertain
      (b) BERT predicts dengue AND confidence < DENGUE_OVERRIDE_CAP
              -- known false-positive for generic symptoms like fever/headache

    Blended probability:
        p_final = alpha * p_BERT + (1-alpha) * p_TF-IDF
    where alpha is dynamic: lower when BERT is less confident.

    Returns
    -------
    disease    : str   -- predicted disease name
    conf       : float -- confidence % of top prediction
    top3       : list  -- [(disease, pct), ...] top-3 predictions
    blend_used : bool  -- True if TF-IDF blend was applied
    """
    model.eval()
    enc = tokenizer(
        str(text), max_length=MAX_LEN, padding='max_length',
        truncation=True, return_tensors='pt'
    )
    with torch.no_grad():
        out = model(
            input_ids=enc['input_ids'].to(DEVICE),
            attention_mask=enc['attention_mask'].to(DEVICE)
        )

    bert_probs = torch.softmax(out.logits, dim=1).cpu().numpy()[0]
    bert_pred  = int(bert_probs.argmax())
    top_conf   = float(bert_probs.max())
    blend_used = False

    # ── TF-IDF disambiguation gate ─────────────────────────────────────
    # Triggers when BERT is uncertain OR when it picks dengue with moderate
    # confidence (the classic 'always dengue' failure for generic symptoms).
    dengue_id   = DISEASE_LABELS['dengue']
    apply_blend = use_tfidf and (
        top_conf < AMBIGUITY_THRESHOLD
        or (bert_pred == dengue_id and top_conf < DENGUE_OVERRIDE_CAP)
    )

    if apply_blend:
        tfidf_scores = tfidf_profiler.score(text)        # (NUM_LABELS,) normalised

        # Dynamic alpha: trust TF-IDF more when BERT is less confident
        dynamic_alpha = min(
            TFIDF_BLEND_ALPHA,
            (top_conf / AMBIGUITY_THRESHOLD) * TFIDF_BLEND_ALPHA
        )
        probs  = dynamic_alpha * bert_probs + (1.0 - dynamic_alpha) * tfidf_scores
        probs /= probs.sum()     # re-normalise
        blend_used = True
    else:
        probs = bert_probs

    pred = int(probs.argmax())
    top3 = [
        (LABEL_NAMES[int(i)], round(float(probs[i]) * 100, 1))
        for i in probs.argsort()[-3:][::-1]
    ]
    return LABEL_NAMES[pred], round(float(probs[pred]) * 100, 1), top3, blend_used


# ── Original clinical test cases ───────────────────────────────────────
test_cases = [
    ('High fever, retro-orbital pain, rash, myalgia. NS1 antigen positive. Platelet 42000. Leucopenia.',
     'dengue'),
    ('Severe unilateral throbbing headache, photophobia, nausea. No fever. Normal CBC. NS1 negative.',
     'migraine'),
    ('High fever, rigor, sweating. Blood smear P. vivax trophozoites. RDT pLDH positive.',
     'malaria'),
    ('Fever 3 weeks, eschar on axilla, lymphadenopathy. Weil-Felix OXK 1:160. IgM ELISA positive.',
     'scrub_typhus'),
    ('Prolonged fever, massive splenomegaly, weight loss. rK39 positive. Haemoglobin 7.2.',
     'kala_azar'),
    ('Breakthrough seizures, EEG spike-wave, on levetiracetam. AED levels subtherapeutic.',
     'epilepsy'),
    ('Progressive memory loss, confusion, wandering, age 74. MMSE 18. Hippocampal atrophy.',
     'dementia'),
    ('Sudden right-sided weakness, aphasia, 2 hours ago. CT no haemorrhage. MRI DWI infarct.',
     'stroke'),
    ('Resting tremor right hand, bradykinesia, rigidity. DaT scan reduced. Levodopa response.',
     'parkinsons'),
    ('Burning feet, numbness, HbA1c 9.2%. Nerve conduction reduced. 10g monofilament positive.',
     'diabetic_neuropathy'),
    ('Polyarthralgia, joint swelling, fever after mosquito bite. IgM ELISA chikungunya positive.',
     'chikungunya'),
    ('Fever, altered consciousness, neck stiffness. CSF IgM JE positive. Thalamic T2 changes.',
     'japanese_encephalitis'),
]

# ── Ambiguous / generic symptom test cases ─────────────────────────
# These previously always returned 'dengue'. Goal: diverse predictions.
ambiguous_cases = [
    ('Fever and headache.',
     'ambiguous — dengue/malaria/scrub'),
    ('High fever, severe headache, muscle aches, fatigue.',
     'ambiguous — dengue/malaria/scrub/chikungunya'),
    ('Fever with joint pain and rash.',
     'ambiguous — dengue/chikungunya'),
    ('Cyclic high fever with rigors and sweating every 48 hours.',
     'malaria (cyclicity clue)'),
    ('Persistent fever 3 weeks, weight loss, enlarged liver and spleen.',
     'kala_azar (duration + splenomegaly)'),
    ('Fever with eschar, swollen lymph nodes, rural scrub habitat exposure.',
     'scrub_typhus (eschar is pathognomonic)'),
    ('Fever with altered consciousness and neck stiffness after mosquito bite.',
     'japanese_encephalitis (CNS signs)'),
]

# ── Run original cases ───────────────────────────────────────────────
print('HealthAI — Clinical Test Cases  (BERT + TF-IDF Blend)')
print('=' * 70)
correct = 0
predicted_classes = set()
for text, expected in test_cases:
    disease, conf, top3, blended = predict(text)
    predicted_classes.add(disease)
    tick = 'CORRECT' if disease == expected else 'WRONG'
    if disease == expected:
        correct += 1
    blend_note = '  [TF-IDF blend]' if blended else ''
    print(f'Input    : {text[:75]}...')
    print(f'Predicted: {disease.upper()} ({conf}%) [{tick}]  Expected: {expected}{blend_note}')
    print(f'Top 3    : {top3}')
    print()

print(f'Demo accuracy : {correct}/{len(test_cases)} ({correct/len(test_cases)*100:.0f}%)')
print(f'Unique classes predicted: {len(predicted_classes)}')
if len(predicted_classes) < 6:
    print('WARNING: fewer than 6 unique classes predicted — may still be collapsing.')
else:
    print('Good diversity in predictions ✓')

# ── Run ambiguous cases ──────────────────────────────────────────────
print()
print('HealthAI — Ambiguous Symptom Test  (TF-IDF Disambiguation)')
print('=' * 70)
print('Goal: diverse predictions across diseases.')
print('All-dengue = bias still present.  <=2 dengue = fix working.\n')
ambig_preds = []
for text, note in ambiguous_cases:
    disease, conf, top3, blended = predict(text)
    ambig_preds.append(disease)
    blend_note = '  [TF-IDF blend]' if blended else ''
    print(f'Input    : {text}')
    print(f'Expected : {note}')
    print(f'Predicted: {disease.upper()} ({conf}%){blend_note}')
    print(f'Top 3    : {top3}')
    print()

dengue_count = ambig_preds.count('dengue')
print(f'Dengue predictions in ambiguous set: {dengue_count}/{len(ambiguous_cases)}')
if dengue_count == len(ambiguous_cases):
    print('WARNING: All ambiguous cases still predict dengue.')
    print('Lower AMBIGUITY_THRESHOLD (try 0.65) or DENGUE_OVERRIDE_CAP (try 0.80).')
elif dengue_count <= 2:
    print('TF-IDF disambiguation working ✔  ambiguous symptoms spread across diseases.')
else:
    print('Partial improvement. Consider lowering AMBIGUITY_THRESHOLD further.')

## Step 15 — Save & package for download


In [ ]:
assert TRAINING_COMPLETE, 'Training did not complete! Run Steps 9-10 first.'
import zipfile
from datetime import datetime

SAVE_DIR = 'healthai_classifier_synthetic'
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

with open(f'{SAVE_DIR}/label_map.json', 'w') as f:
    json.dump(LABEL_NAMES, f, indent=2)

with open(f'{SAVE_DIR}/model_info.json', 'w') as f:
    json.dump({
        'version':         'synthetic-v2-bulletproof',
        'base_model':      MODEL_NAME,
        'training_data':   'healthai_synthetic_training_set_7200',
        'train_samples':   len(train_df),
        'val_samples':     len(val_df),
        'test_samples':    len(test_df),
        'accuracy':        round(acc * 100, 2),
        'f1_macro':        round(f1m, 4),
        'f1_weighted':     round(f1w, 4),
        'best_p2_epoch':   best_epoch,
        'phase1_epochs':   PHASE1_EPOCHS,
        'phase2_epochs':   PHASE2_EPOCHS,
        'diseases':        list(DISEASE_LABELS.keys()),
        'trained':         datetime.now().isoformat(),
    }, f, indent=2)

ZIP = 'healthai_model_synthetic_v2.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in os.listdir(SAVE_DIR):
        zf.write(f'{SAVE_DIR}/{fn}', fn)
    for fn in ['confusion_matrix.png', 'training_curves.png', 'best_model.pt']:
        if os.path.exists(fn): zf.write(fn, fn)

size_mb = os.path.getsize(ZIP) / 1e6
print(f'Saved: {ZIP}  ({size_mb:.1f} MB)')
print('Download: Files panel → right-click → Download')
print()
print('=' * 55)
print('TRAINING COMPLETE')
print('=' * 55)
print(f'Accuracy  : {acc*100:.2f}%')
print(f'F1 macro  : {f1m:.4f}')
print(f'Best epoch: Phase2 epoch {best_epoch}')
print('=' * 55)
